[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: actividad «Hello world»

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

Este cuaderno resuelve la actividad del cuaderno `02_actividad_hello_world.ipynb`. Requiere una cuenta personal de *GitHub* y el *CLI* `gh` con la sesión iniciada (`gh auth login`) para las celdas con etiqueta.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Crear un repositorio en GitHub

Crear, en la cuenta personal, un repositorio **público** llamado `demo-github-actions`.

El cuaderno original lo resuelve con la interfaz web (botón `New`). La solución equivalente con el *CLI* es un solo comando. La opción `--add-readme` crea el primer *commit* y, con él, la rama `main`, necesaria para el siguiente ejercicio.

La siguiente celda creará el repositorio público `demo-github-actions` en la cuenta con la sesión iniciada en `gh` y lo clonará en la carpeta personal (`$HOME`), fuera del repositorio del curso.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}"
gh auth status
gh repo create demo-github-actions --public --add-readme --clone
cd "${HOME}/demo-github-actions"
gh repo view --json name,visibility,defaultBranchRef

**Salida esperada.** `gh auth status` indica la cuenta con la sesión iniciada. `gh repo create` imprime la dirección del repositorio nuevo y el último comando muestra un objeto *JSON* con `"name": "demo-github-actions"`, `"visibility": "PUBLIC"` y la rama por defecto `main`.

## Ejercicio 2. Crear un workflow simple

En la pestaña `Actions` del repositorio, usar la plantilla `Simple workflow`. Se crea el archivo `.github/workflows/blank.yml`, que se guarda con un *commit* y desencadena su primera ejecución.

La plantilla es un texto *YAML*. La siguiente celda lo define en una variable de *Python* para poder comprobarlo antes de enviarlo a *GitHub*. Las acciones se fijan por el *SHA* completo de su *commit*, como se hace en el resto del curso.

La siguiente celda definirá la variable `BLANK` con el contenido de `blank.yml`: un *workflow* llamado `CI` con tres disparadores (`push` y `pull_request` sobre `main`, y `workflow_dispatch`) y un *job* `build` de tres pasos.

In [ ]:
BLANK = """\
name: CI

on:
  push:
    branches: ["main"]
  pull_request:
    branches: ["main"]
  workflow_dispatch:

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@3d3c42e5aac5ba805825da76410c181273ba90b1 # v7.0.1

      - name: Run a one-line script
        run: echo Hello, world!

      - name: Run a multi-line script
        run: |
          echo Add other actions to build,
          echo test, and deploy your project.
"""

La siguiente celda convertirá `BLANK` en un diccionario con `cargar_workflow()` y comprobará con `assert` su estructura: nombre, disparadores, tipo de *runner*, número de pasos y acciones fijadas por *SHA*. Si todo es correcto, no imprime nada salvo la línea final. Obsérvese que la clave `on` se lee con la función auxiliar, porque *PyYAML* interpreta `on` como el valor booleano `True`.

In [ ]:
wf = u.cargar_workflow(BLANK)

assert wf["name"] == "CI"
assert set(wf["on"]) == {"push", "pull_request", "workflow_dispatch"}
assert wf["on"]["push"]["branches"] == ["main"]
assert wf["jobs"]["build"]["runs-on"] == "ubuntu-latest"
assert len(u.pasos(wf, "build")) == 3
assert u.acciones_sin_fijar(wf) == []
print("Estructura correcta:", list(wf["jobs"]), "con", len(u.pasos(wf, "build")), "pasos")

La siguiente celda guardará `BLANK` como `.github/workflows/blank.yml` dentro de un repositorio *Git* temporal y hará un *commit* con una identidad de ejemplo. Es el ensayo local de lo que hace el botón `Commit new file`; no se contacta con *GitHub*.

In [ ]:
import subprocess

TEMPORAL = u.directorio_temporal()
REPO = TEMPORAL / "demo-github-actions"
(REPO / ".github" / "workflows").mkdir(parents=True)
(REPO / ".github" / "workflows" / "blank.yml").write_text(BLANK, encoding="utf-8")

def git(*argumentos):
    return subprocess.run(["git", "-C", str(REPO), *argumentos], check=True, capture_output=True, text=True).stdout

git("init", "--quiet", "--initial-branch=main")
git("add", ".github/workflows/blank.yml")
git("-c", "user.name=Persona Ejemplo", "-c", "user.email=persona@example.com",
    "commit", "--quiet", "--message", "Create blank.yml")
print(git("log", "--format=%an <%ae> | %s"), end="")
print(git("ls-files"), end="")

La siguiente celda llevará `blank.yml` al clon real del repositorio, hará el *commit* y lo enviará a `origin`. Ese `push` a `main` desencadena la primera ejecución del *workflow*.

In [ ]:
import subprocess
from pathlib import Path

clon = Path.home() / "demo-github-actions"
destino = clon / ".github" / "workflows" / "blank.yml"
destino.parent.mkdir(parents=True, exist_ok=True)
destino.write_text(BLANK, encoding="utf-8")
for comando in (
    ["git", "-C", str(clon), "add", ".github/workflows/blank.yml"],
    ["git", "-C", str(clon), "commit", "--message", "Create blank.yml"],
    ["git", "-C", str(clon), "push", "origin", "main"],
):
    subprocess.run(comando, check=True)

**Salida esperada.** `git commit` informa de un archivo nuevo y `git push` termina sin errores. Si `git commit` pide la identidad, se configura una vez con `git config user.name` y `git config user.email`. En la pestaña `Actions` aparece una ejecución del *workflow* `CI` que pasa de «en curso» a «correcta».

## Ejercicio 3. Esperar la primera ejecución y lanzar una manual

Comprobar que la ejecución desencadenada por el *commit* termina correctamente. Después ejecutar el *workflow* a mano, con el disparador `workflow_dispatch`, y esperar a que termine.

Antes de usar la nube conviene ensayar localmente qué hará el *job* `build`. `simular_job()` ejecuta con `bash` los pasos `run`; el paso `uses` (el *checkout*) no se ejecuta en el ensayo.

La siguiente celda ensayará el *job* `build` de `blank.yml` en una carpeta temporal y mostrará, por cada paso, su nombre, su código de salida y su salida. Se espera `Hello, world!` y las dos líneas del paso de varias líneas.

In [ ]:
resultados = u.simular_job(wf, "build", contextos={}, directorio=TEMPORAL)
u.mostrar_resultados(resultados)

salidas = {nombre: salida for nombre, salida, _ in resultados}
assert salidas["Run a one-line script"] == "Hello, world!"
assert salidas["Run a multi-line script"].splitlines() == [
    "Add other actions to build,",
    "test, and deploy your project.",
]
assert all(codigo == 0 for _, _, codigo in resultados)

La siguiente celda seguirá la ejecución del *workflow* en *GitHub*: lanza una ejecución manual, espera a que termine y lista las dos últimas. El evento de la manual es `workflow_dispatch`; el de la primera es `push`.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}/demo-github-actions"
gh workflow run blank.yml --ref main
sleep 5
ID="$(gh run list --workflow blank.yml --event workflow_dispatch --limit 1 --json databaseId --jq '.[0].databaseId')"
gh run watch "${ID}" --exit-status
gh run list --workflow blank.yml --limit 2

**Salida esperada.** `gh run watch` muestra el *job* `build` y sus pasos con una marca de éxito, y termina con código 0. `gh run list` presenta dos filas con estado `completed` y conclusión `success`: una con el evento `push` y otra con `workflow_dispatch`.

## Ejercicio 4. Explorar el proceso de un workflow

Abrir el resumen de la ejecución manual y reconocer qué *jobs* se ejecutaron, qué pasos tiene el *job* `build` y qué hace *GitHub Actions* al empezar y al terminar.

En el registro de un *job* siempre aparece primero la preparación del entorno (`Set up job`), después cada paso del archivo en orden, los pasos posteriores de las acciones que los tienen (por ejemplo, `Post Run actions/checkout`) y, al final, `Complete job`, que limpia el entorno.

La siguiente celda calculará, a partir de `blank.yml`, la lista de pasos que debe mostrar el registro del *job* `build`: los dos pasos propios de *GitHub Actions* al principio y al final, y los pasos del archivo entre ellos. Una acción con `uses` añade además un paso `Post` al terminar.

In [ ]:
esperado = ["Set up job"]
posteriores = []
for paso in u.pasos(wf, "build"):
    if "uses" in paso:
        accion = paso["uses"].split("@")[0]
        esperado.append(f"Run {accion}")
        posteriores.insert(0, f"Post Run {accion}")
    else:
        esperado.append(paso["name"])
esperado += posteriores + ["Complete job"]

for numero, nombre in enumerate(esperado, start=1):
    print(f"{numero}. {nombre}")
assert len(wf["jobs"]) == 1 and "build" in wf["jobs"]

La siguiente celda mostrará, desde la terminal, el registro de la última ejecución con `gh run view --log`, para compararlo con la lista anterior.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}/demo-github-actions"
ID="$(gh run list --workflow blank.yml --limit 1 --json databaseId --jq '.[0].databaseId')"
gh run view "${ID}" --json jobs --jq '.jobs[] | {job: .name, pasos: [.steps[].name]}'
gh run view "${ID}" --log | head -n 40

**Salida esperada.** Un único *job*, `build`, con los pasos `Set up job`, `Run actions/checkout@…`, `Run a one-line script`, `Run a multi-line script`, `Post Run actions/checkout@…` y `Complete job`. En el registro se lee `Hello, world!` bajo el paso de una línea.

## Verificación final

La lista de comprobaciones es la siguiente:

* `blank.yml` es *YAML* válido, con tres disparadores y un *job* de tres pasos.
* El ensayo local produjo `Hello, world!`.
* En *GitHub*, las dos ejecuciones terminaron en `success`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2027.</p>